In [ ]:
import os

# 1. Set the custom cache directory via the environment variable
# This must be done BEFORE importing Keras/TensorFlow to ensure it applies
os.environ["KERAS_HOME"] = os.path.abspath("./my_keras_cache")

import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

# =====================================================================
# STAGE A: Loading and Preprocessing the Image Data
# =====================================================================
print("--- Stage A: Loading and preparing the CIFAR-10 dataset ---")

# FIXED: Removed the unsupported cache_dir keyword argument.
# Keras automatically uses the environment variable set above.
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize pixel values from range [0, 255] to range [0, 1] for stable gradients
x_train, x_test = (
    x_train.astype("float32") / 255.0,
    x_test.astype("float32") / 255.0,
)

# Convert integer class vectors to binary class matrices (One-Hot Encoding)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)


# =====================================================================
# STAGE B: Defining the Model's Architecture
# =====================================================================
print("\n--- Stage B: Defining the CNN model architecture ---")
model = Sequential(
    [
        # First Convolutional Block: Extracts low-level features like edges
        Conv2D(
            32, (3, 3), activation="relu", padding="same", input_shape=(32, 32, 3)
        ),
        MaxPooling2D((2, 2)),
        # Second Convolutional Block: Extracts mid-level features like shapes
        Conv2D(64, (3, 3), activation="relu", padding="same"),
        MaxPooling2D((2, 2)),
        # Third Convolutional Block: Extracts high-level features like complex textures
        Conv2D(128, (3, 3), activation="relu", padding="same"),
        MaxPooling2D((2, 2)),
        # Flatten: Converts the 3D feature maps to a 1D vector for classification
        Flatten(),
        # Fully Connected (Dense) Layers for final classification
        Dense(128, activation="relu"),
        # Output layer with 10 neurons and softmax activation for probability distribution
        Dense(10, activation="softmax"),
    ]
)

# Print structural layout of the neural network
model.summary()

# Compile the model with Adam optimizer and Categorical Crossentropy Loss
print("\nCompiling the model...")
model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)


# =====================================================================
# STAGE C: Training the Model
# =====================================================================
print("\n--- Stage C: Training the model ---")
# Training for 15 epochs with a batch size of 64
history = model.fit(
    x_train, y_train, epochs=15, batch_size=64, validation_data=(x_test, y_test)
)


# =====================================================================
# STAGE D: Estimating the Model's Performance
# =====================================================================
print("\n--- Stage D: Estimating the model's performance ---")
# Evaluate on unseen validation dataset
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nFinal Test accuracy: {test_acc*100:.2f}%")

# Make a standalone prediction to test execution logic
print("\nMaking a prediction on a single test image...")
class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Expand dimensions from (32, 32, 3) to (1, 32, 32, 3) to represent a batch size of 1
img = np.expand_dims(x_test[0], axis=0)
prediction = model.predict(img)

# Fetch index coordinates of the highest matching probability values
predicted_class = np.argmax(prediction)
true_class = np.argmax(y_test[0])

print(f"The model predicts the image is a '{class_names[predicted_class]}'.")
print(f"The true class label is a '{class_names[true_class]}'.")


--- Stage A: Loading and preparing the CIFAR-10 dataset ---
A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce so we will re-download the data.
 57147392/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:05:21 35us/step